# AprioriTID - Demo

Sổ tay này trình diễn ba phần chính của đồ án:

1. **Cài đặt cơ bản** --- chạy `apriori_tid` trên một bộ dữ liệu toy rất nhỏ, từng bước một.
2. **Kiểm thử đơn vị trên 5 bộ dữ liệu toy** --- kiểm tra đúng đắn so với kết quả đã tính tay.
3. **Ứng dụng trên bộ dữ liệu Retail** --- phân tích giỏ hàng với các luật kết hợp.

Toàn bộ mã nguồn viết bằng Julia 1.9+. Chạy các cell từ trên xuống dưới, tại thư mục gốc của đồ án.

In [ ]:
# Load the algorithm and utilities from src/
const PROJECT_ROOT = dirname(@__DIR__)
include(joinpath(PROJECT_ROOT, "src", "structures.jl"))
include(joinpath(PROJECT_ROOT, "src", "utils.jl"))
include(joinpath(PROJECT_ROOT, "src", "algorithm", "apriori_gen.jl"))
include(joinpath(PROJECT_ROOT, "src", "algorithm", "apriori_tid.jl"))
println("AprioriTID loaded")

## 1. Cài đặt cơ bản

Ta chạy `apriori_tid` trên file `data/toy/example1.txt`, đúng là bộ dữ liệu của Ví dụ 1 trong báo cáo (5 giao dịch, 5 item, `minsup = 2`). Kết quả mong đợi là 11 tập phổ biến, trùng khớp với lần chạy tay trong Chương 2.

In [ ]:
ex1_path = joinpath(PROJECT_ROOT, "data", "toy", "example1.txt")
transactions = read_spmf(ex1_path)
println("Transactions:")
for (tid, t) in enumerate(transactions)
    println("  T", tid, " = ", t)
end
println("|D| = ", length(transactions))

In [ ]:
# Run AprioriTID at minsup = 2
results = apriori_tid(transactions, 2)

# Group by |F|
by_len = Dict{Int, Vector{Tuple{Vector{Int}, Int}}}()
for (iset, sup) in results
    push!(get!(by_len, length(iset), Tuple{Vector{Int}, Int}[]), (iset, sup))
end

println("Total frequent itemsets: ", length(results))
for k in sort(collect(keys(by_len)))
    println("\nL_", k, " (", length(by_len[k]), " itemsets):")
    for (iset, sup) in sort(by_len[k], by = x -> x[1])
        println("  ", iset, "  #SUP: ", sup)
    end
end

In [ ]:
# Show that the `optimized=false` (basic) path gives the same frequent itemsets
basic_results = apriori_tid(transactions, 2; optimized=false)
opt_set   = Set((iset, sup) for (iset, sup) in results)
basic_set = Set((iset, sup) for (iset, sup) in basic_results)
println("Basic == Optimized output: ", opt_set == basic_set)

## 2. Kiểm thử đơn vị trên 5 bộ dữ liệu toy

Năm bộ dữ liệu toy nằm trong thư mục `data/toy/`, mỗi bộ kiểm tra một tình huống biên khác nhau:

| File                      | Tình huống kiểm tra                                          |
|---------------------------|--------------------------------------------------------------|
| `example1.txt`            | Ví dụ 1 chuẩn (5 × 5, minsup = 2)                            |
| `example2.txt`            | Trường hợp dày đặc, $\bar{C}_2$ lớn hơn $D$                  |
| `test_single_item.txt`    | Các giao dịch chỉ có đúng một item                           |
| `test_empty_result.txt`   | Không item nào đạt `minsup = 3` --- đầu ra phải rỗng         |
| `test_all_frequent.txt`   | 3 giao dịch hoàn toàn giống nhau --- mọi tập con đều phổ biến |

Với mỗi bộ, ta so sánh kết quả của `apriori_tid` với đáp án tính tay.

In [ ]:
function to_set(res)
    return Set((sort(iset), sup) for (iset, sup) in res)
end

cases = [
    ("example1.txt",          2, 11),
    ("example2.txt",          2, nothing),  # expected count computed from run
    ("test_single_item.txt",  2, nothing),
    ("test_empty_result.txt", 3, 0),
    ("test_all_frequent.txt", 2, 7),        # 2^3 - 1 = 7 non-empty subsets of {1,2,3}
]

println("Running toy-dataset unit checks...\n")
for (fname, minsup, expected_n) in cases
    path = joinpath(PROJECT_ROOT, "data", "toy", fname)
    trans = read_spmf(path)

    opt   = apriori_tid(trans, minsup; optimized=true)
    basic = apriori_tid(trans, minsup; optimized=false)

    agree = to_set(opt) == to_set(basic)
    n     = length(opt)

    tag = agree ? "OK" : "FAIL"
    expected_str = expected_n === nothing ? "-"       : string(expected_n)
    count_ok     = expected_n === nothing ? true      : (n == expected_n)
    count_tag    = count_ok ? "OK" : "FAIL"

    println(rpad(fname, 24),
            " minsup=", lpad(minsup, 2),
            "  |D|=", lpad(length(trans), 3),
            "  #itemsets=", lpad(n, 3),
            "  expected=", rpad(expected_str, 4),
            "  basic==opt: ", tag,
            "  count: ", count_tag)
end

In [ ]:
# Compare Example 1 against the SPMF reference file if available
spmf_ref = joinpath(PROJECT_ROOT, "data", "toy", "spmf_reference", "example1.txt")
if isfile(spmf_ref)
    ref = read_spmf_output(spmf_ref)
    ours = apriori_tid(read_spmf(joinpath(PROJECT_ROOT, "data", "toy", "example1.txt")), 2)
    println("SPMF reference matches our output: ", to_set(ref) == to_set(ours))
else
    println("SPMF reference file not found: ", spmf_ref)
end

## 3. Ứng dụng --- Phân tích giỏ hàng trên Retail

Giờ ta áp dụng thuật toán lên `data/benchmark/retail.txt` (88\,162 giao dịch) và sinh các luật kết hợp $X \Rightarrow Y$ kèm ba độ đo kinh điển:

$$\mathrm{support} = \frac{|\{t : X \cup Y \subseteq t\}|}{|D|}, \qquad \mathrm{confidence} = \frac{\mathrm{sup}(X \cup Y)}{\mathrm{sup}(X)}, \qquad \mathrm{lift} = \frac{\mathrm{confidence}}{\mathrm{sup}(Y) / |D|}$$

Mã sinh luật nằm trong `application/market_basket.jl`. Xem Chương 5 của báo cáo để đọc phần thảo luận đầy đủ.

In [ ]:
include(joinpath(PROJECT_ROOT, "application", "market_basket.jl"))

retail_path = joinpath(PROJECT_ROOT, "data", "benchmark", "retail.txt")
retail = read_spmf(retail_path)
n_trans = length(retail)
println("|D| = ", n_trans, " transactions loaded")

In [ ]:
# Step 1: mine frequent itemsets (this is the expensive step)
minsup = 500   # absolute, ~0.57%
t_fim = @elapsed freq = apriori_tid(retail, minsup)
println("AprioriTID: ", length(freq), " frequent itemsets in ",
        round(t_fim * 1000, digits=1), " ms")
println("Itemsets with |F| >= 2: ", count(x -> length(x[1]) >= 2, freq))

In [ ]:
# Step 2: generate association rules at several minconf thresholds.
# Rule generation is O(sum 2^|F|) dict lookups — essentially free compared to FIM.
for minconf in (0.30, 0.50, 0.70, 0.90)
    t = @elapsed rules = generate_rules(freq, n_trans, minconf)
    println("minconf=", minconf,
            "  -> ", lpad(length(rules), 4), " rules",
            "  (", round(t * 1000, digits=2), " ms)")
end

In [ ]:
# Full rule set at minconf=0.30
rules = generate_rules(freq, n_trans, 0.30)

function show_rule(r)
    println("  {", join(r.antecedent, ","),
            "} => {", join(r.consequent, ","), "}  ",
            "sup=",  round(r.support,    digits=4),
            "  conf=", round(r.confidence, digits=3),
            "  lift=", round(r.lift,       digits=2))
end

println("Top 10 by confidence:")
foreach(show_rule, top_rules_by(rules, :confidence; n=10))

println("\nTop 10 by lift:")
foreach(show_rule, top_rules_by(rules, :lift; n=10))

In [ ]:
# Persist the full rule set to CSV for downstream use
out_dir = joinpath(PROJECT_ROOT, "experiment_results")
isdir(out_dir) || mkpath(out_dir)
out_path = joinpath(out_dir, "retail_rules.csv")
write_rules_csv(out_path, rules)
println("Wrote ", length(rules), " rules to ", out_path)

### Các điểm chính rút ra

- **Item 39 lấn át bảng xếp hạng theo confidence.** Mọi luật top-confidence đều có hậu đề là $\{39\}$ vì item 39 xuất hiện trong khoảng 57\,\% giao dịch. Xếp hạng chỉ theo confidence dễ đánh lừa.
- **Lift phơi bày quan hệ đồng xuất hiện thực sự.** Các cặp $\{16011, 16012\}$ và $\{271, 272\}$ có support rất nhỏ nhưng lift $> 16$ --- chúng gần như không bao giờ xuất hiện rời nhau.
- **Tính bất đối xứng rất quan trọng.** $16012 \Rightarrow 16011$ đạt confidence 0.97 trong khi $16011 \Rightarrow 16012$ chỉ đạt 0.50 --- một tín hiệu kinh điển của quan hệ "phụ kiện → mặt hàng chính".
- **FIM mới là nút thắt cổ chai.** Bước sinh luật chỉ mất vài mili-giây; lời gọi AprioriTID chiếm trọn thời gian chạy pipeline. Một lần chạy FIM đủ để phục vụ nhiều truy vấn phân tích luật khác nhau về sau.